In [1]:
import pandas as pd

In [40]:
pd.set_option('display.max_colwidth', None)
#pd.reset_option('display.max_colwidth')

In [63]:
episodes = pd.read_csv('../data/episodes_with_id.csv')
episodes_filtered = episodes[episodes['numMainSpeakers'] == 3]
episodes_filtered
episodes_filtered[['epTitle', 'mp3url','episode_id']].head(50)

,epTitle,mp3url,episode_id
0,Bringing Back Our Sense of Community,https://anchor.fm/s/11abe148/podcast/play/15344549/https%3A%2F%2Fd3ctxlq1ktw2nl.cloudfront.net%2Fproduction%2F2020-5-17%2F83194249-44100-2-bc23c65d66c68.mp3,1
3,OWR 203,https://anchor.fm/s/126c0978/podcast/play/15840996/https%3A%2F%2Fd3ctxlq1ktw2nl.cloudfront.net%2Fstaging%2F2020-07-21%2Fccb7a609626d52cfe97eebc50d050859.m4a,4
21,"Daily Readings Only - Tuesday June 2, 2020 Blackout Tuesday",https://feeds.soundcloud.com/stream/832836613-ddornerjr-daily-readings-only-tuesday.mp3,22
34,What is 'Real' Christianity?,https://anchor.fm/s/fc0c4ac/podcast/play/14947756/https%3A%2F%2Fd3ctxlq1ktw2nl.cloudfront.net%2Fproduction%2F2020-5-9%2F80799873-32000-1-349c8436f3b94.mp3,35
37,Cung Le exposes UFC & Family Law courts,https://anchor.fm/s/2558474/podcast/play/14609920/https%3A%2F%2Fd3ctxlq1ktw2nl.cloudfront.net%2Fstaging%2F2020-12-14%2F616c589d7785dc429b8c3e913720b2df.m4a,38
39,246) William Defebaugh: Exploring the intricate balance between the flourishing and decay of life,https://podcasts.captivate.fm/media/885509ef-e1b8-47ed-971c-239dc0615202/246-exploring-the-intricate-balance-between-the-flourishing-and.mp3,40
43,S1 Ep8: Appropriation in a Global Food World,https://chrt.fm/track/8B7A46/traffic.megaphone.fm/PODS5540178173.mp3?updated=1668598911,44
50,"PSP Let's Talk: Yohannes Berhane, Jake Wenstrup",https://anchor.fm/s/1f999408/podcast/play/14603033/https%3A%2F%2Fd3ctxlq1ktw2nl.cloudfront.net%2Fstaging%2F2020-5-1%2F78649033-44100-2-417e51e41b09f.m4a,51
76,Black Lives Matter: The View from Ireland,https://www.buzzsprout.com/886948/4360184-black-lives-matter-the-view-from-ireland.mp3,77
83,Medical Students Demand an Anti-Racist Education,https://static1.squarespace.com/static/5e34a702359e3e28ffa093ef/t/5ef03a58e69a0b0f27d517ab/1592802066145/Well+Rounded_Ep+10_Betial+Asmerom.mp3,84


In [64]:
# paragraph_turns = pd.read_csv('../data/paragraph_turns_with_emotions.csv')
# paragraph_turns[paragraph_turns['episode_id'] == 24].tail(10)

In [71]:
mergedDf = pd.read_csv('../data/speaker_turns.csv')

In [74]:
filtered_df = mergedDf[mergedDf['mp3url'] == 'https://traffic.megaphone.fm/GOAL3562780589.mp3']
filtered_df.head()

,mfcc1_sma3Mean,mfcc2_sma3Mean,mfcc3_sma3Mean,mfcc4_sma3Mean,F0semitoneFrom27_5Hz_sma3nzMean,F1frequency_sma3nzMean,turnText,speaker,startTime,endTime,duration,mp3url,turnCount,inferredSpeakerRole,inferredSpeakerName


In [62]:
# Assign text colors to speakers
speaker_colors = {
    'SPEAKER_00': '#E66100',  # Red-Orange
    'SPEAKER_01': '#5D3A9B',  # Dodger Blue
    'SPEAKER_00, SPEAKER_01': '#999999',  # Green
    'SPEAKER_01, SPEAKER_00': '#999999',  # Blue-Violet
}

# Build HTML content
html_content = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Transcript</title>
    <style>
        body { font-family: Arial, sans-serif; line-height: 1.5; }
    </style>
</head>
<body>
<p>
"""

for _, row in filtered_df.iterrows():
    color = speaker_colors.get(row['speakers'], '#000000')  # Default black if no match
    html_content += f'<span style="color: {color};">{row["content"]} </span>'

html_content += "</p>\n</body></html>"

# Save to an HTML file
with open("../data/transcript_color.html", "w", encoding="utf-8") as f:
    f.write(html_content)

print("HTML file saved as transcript.html")

HTML file saved as transcript.html


In [57]:
mergedDf.rename(columns={
    'mfcc1_sma3Mean': 'mfcc1_sma3',
    'mfcc2_sma3Mean': 'mfcc2_sma3',
    'mfcc3_sma3Mean': 'mfcc3_sma3Mean',
    'mfcc4_sma3Mean': 'mfcc4_sma3Mean',
    'F0semitoneFrom27_5Hz_sma3nzMean': 'F0semitoneFrom27.5Hz_sma3nz',
    'F1frequency_sma3nzMean': 'F1frequency_sma3nz',
    'speaker': 'speakers',
    'startTime': 'start',
    'endTime': 'end',
    'turnText': 'content'
}, inplace=True)

Pick an episode.

In [44]:
def getColoredTranscript(mergedDf): 
    #from colorama import Fore
    #foreColors = [Fore.RED, Fore.GREEN, Fore.YELLOW, Fore.BLUE, Fore.MAGENTA, Fore.CYAN]
    colList = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#a65628','#f781bf']

    uSpeakers = set([speakerList[0] for speakerList in mergedDf["speakers"] if len(speakerList) > 0])
    numSpeakers = len(uSpeakers) 
    colDict = dict(zip(uSpeakers, colList[:numSpeakers]))
    colDict["NONE"] = "#000000"
    colDict["MULT"] = '#999999'


    #highlight transcript colors 
    pastSpeakList = "NONE"
    currText = ""
    allText = ""

    for i, row in mergedDf.iterrows(): 
        word = row["content"]
        speakList = row["speakers"]

        """ 
        if len(speakers) > 0: 
            speakList = speakers[-1]
        else: 
        """
        if len(speakList) == 0: 
            speakList = ["NONE"]

        if speakList != pastSpeakList: 
            if len(pastSpeakList) > 1: 
                allText += f'<font color = "{colDict["MULT"]}">{currText}</font>'
            else:  
                allText += f'<font color = "{colDict[pastSpeakList[0]]}">{currText}</font>'

            currText = ""

        if word == word and word != None: 
            currText += word
        pastSpeakList = speakList

    return allText

In [45]:
getColoredTranscript(filtered_df)

'<font color = "#999999"></font><font color = "#999999"> Thank you for listening to this episode of Changes Big and Small. This is your host, Damien. Each week I share research or interview guests to help you make changes in your own life. Currently I am focused on a special series exploring equality, justice and anti-racism. Today I am speaking with Brian Summers who lives in New Jersey. Brian Summers has honed his craft as a portrait photographer capturing classical images of guests of the hip hop and culture podcast, The Combat Jack Show. In addition to hosting two of his own weekly photography based podcast, we are getting better and shooting with shooters. Since moving to New York in 2013, Summers has worked on personal photography projects that eventually led to art exhibitions in New York and Washington, DC. After taking the stage at a creative morning\'s New York event, Brian was bitten by the speaking bug. Since then, he has used his experiences in media, fine arts and persona

In [46]:
def save_transcript_as_html(transcript, filename="../data/transcript_color.html"):
    with open(filename, "w", encoding="utf-8") as f:
        f.write("<html><body>" + transcript + "</body></html>")
    print(f"Transcript saved as {filename}")

html_transcript = getColoredTranscript(filtered_df)
save_transcript_as_html(html_transcript)

Transcript saved as ../data/transcript_color.html
